In [2]:
import time

In [4]:
from Bio import Entrez

def search(query):
    Entrez.email = 'your.email@example.com'
    handle = Entrez.esearch(db='pubmed', 
                            sort='relevance', 
                            retmax='2000',
                            retmode='xml',
                            term=query)
    results = Entrez.read(handle)
    return results

def fetch_details(id_list):
    ids = ','.join(id_list)
    Entrez.email = 'your.email@example.com'
    handle = Entrez.efetch(db='pubmed',
                           retmode='xml',
                           id=ids)
    results = Entrez.read(handle)
    return results

In [43]:
# search_term = '(("2020"[Date - Publication] : "2022"[Date - Publication])) AND ("Progress in retinal and eye research"[Journal])'
search_term = '(("2023"[Date - Publication] : "2023"[Date - Publication])) AND ("Annual review of vision science"[Journal])'#Pubmed advanced search journal

results = search(search_term) # or any query you like
id_list = results['IdList'] # list of UIDs
print("Got", len(id_list), "papers")
chunk_size = 50 # whatever you like  

all_papers = []
for chunk_i in range(0, len(id_list), chunk_size):
    print('Getting details for papers', chunk_i, 'to', (chunk_i+chunk_size))
    chunk = id_list[chunk_i:chunk_i + chunk_size]
    try: 
        papers = fetch_details(chunk)
        for i, paper in enumerate(papers['PubmedArticle']):
            all_papers.append(paper)
    except: # occasionally a chunk might annoy your parser
        print('There was an error!')
        pass
    
    time.sleep(1)

Got 23 papers
Getting details for papers 0 to 50


In [38]:
title_list= []
first_author_list = []
last_author_list = []
journal_list = []
pubdate_year_list = []



In [39]:
for i, paper in enumerate(all_papers[:2000]):
    title_list.append("{}) {}".format(i+1, paper['MedlineCitation']['Article']['ArticleTitle']))
    details = paper['MedlineCitation']['Article']
    if 'AuthorList' in details and len(details['AuthorList']) > 0:
        pubdate_year_list.append(details['Journal']['JournalIssue']['PubDate']['Year'])
        journal_list.append(details['Journal']['Title'])
        if 'ForeName' in details['AuthorList'][0] and 'LastName' in details['AuthorList'][0]:
            first_author_list.append(details['AuthorList'][0]['ForeName'] + ' ' + details['AuthorList'][0]['LastName'])
        else:
            first_author_list.append('N/A')

        # Check if the last AuthorList has elements
        if 'ForeName' in details['AuthorList'][-1] and 'LastName' in details['AuthorList'][-1]:
            last_author_list.append(details['AuthorList'][-1]['ForeName'] + ' ' + details['AuthorList'][-1]['LastName'])
        else:
            last_author_list.append('N/A')
        
      
        
      
        
       

In [446]:
print(details['AuthorList'][0]['ForeName'] + ' ' + details['AuthorList'][0]['LastName'])

all_papers[1]



KeyError: 'AuthorList'

In [40]:
import csv
import pandas as pd
import numpy as np


data = pd.DataFrame(list(zip(
    title_list, first_author_list , last_author_list, journal_list, pubdate_year_list
    )), 
    columns=[
             'Title', 'First Author', 'Last Author', 'Journal', 'Year'
             ])

In [41]:
data
len(data)


279

In [42]:
df = pd.DataFrame(data)

# Save the dataframe to a CSV file
df.to_csv('AfterCovidAO.csv', index=False)

In [70]:
# After all datasets have been added together in r

import csv

# Function to separate the first and last names or handle "N/A" values
# Function to separate the first and last names or handle "N/A" values
def first_separate_name(name):
    if name.lower() == 'n/a':
        return 'N/A', 'N/A'
    
    # Split the name by a comma and strip whitespace from both parts
    parts = name.split(',')
    
    if len(parts) >= 1:
        name_part = parts[0].strip()
        words = name_part.split()

        if len(words) >= 2:
            first_first_name = words[0]
            first_last_name = words[-1]
        else:
            first_first_name, first_last_name = name_part, ''
        
        return first_first_name, first_last_name
    else:
        return 'N/A', 'N/A'
    
def last_separate_name(name):
    if name.lower() == 'n/a':
        return 'N/A', 'N/A'
    
    # Split the name by a comma and strip whitespace from both parts
    parts = name.split(',')
    
    if len(parts) >= 1:
        name_part = parts[0].strip()
        words = name_part.split()

        if len(words) >= 2:
            last_first_name = words[0]
            last_last_name = words[-1]
        else:
            last_first_name, last_last_name = name_part, ''
        
        return last_first_name, last_last_name
    else:
        return 'N/A', 'N/A'

# Input and output file paths
input_file_path = 'During_Covid_Pubmed.csv'# Input file name
output_file_path = 'During_Covid_Pubmed_Sep.csv'# Output file name

# Read the input CSV file and write to the output file
with open(input_file_path, 'r') as input_file, open(output_file_path, 'w', newline='') as output_file:
    # Create CSV reader and writer objects
    csv_reader = csv.DictReader(input_file)
    fieldnames = csv_reader.fieldnames + ['First Author First Name', 'First Author Last Name','Last Author First Name', 'Last Author Last Name']
    csv_writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    csv_writer.writeheader()

    # Process and write the data
    # Process and write the data
    for row in csv_reader:
        first_author_name = row['First.Author']
        first_first_name, first_last_name = first_separate_name(first_author_name)
        row['First Author First Name'] = first_first_name
        row['First Author Last Name'] = first_last_name
        
   
        last_author_name = row['Last.Author']
        last_first_name, last_last_name = last_separate_name(last_author_name)
        row['Last Author First Name'] = last_first_name
        row['Last Author Last Name'] = last_last_name
        csv_writer.writerow(row)


print("CSV processing complete. Output written to", output_file_path)


CSV processing complete. Output written to During_Covid_Pubmed_Sep.csv
